In [1]:
import pandas as pd
import numpy as np
import faiss
import pickle
import random

from sklearn.cluster import KMeans
import scipy.sparse as sp

from sklearn.metrics.pairwise import cosine_similarity

In [2]:
n_anchors = 40
n_random_anchors = 10

n_e5_candidates = 20
n_tfidf_candidates = 20
n_random_candidates = 10

In [3]:
index_path = "../data/cache/cache_index.faiss"
meta_path = "../data/cache/cache_meta.pkl"
vacancies_path = "../data/processed/cleaned_vacancies.csv"

vacancies_df = pd.read_csv(vacancies_path)
index = faiss.read_index(index_path)

with open("../data/cache/cache_meta.pkl", "rb") as f:
    meta = pickle.load(f)

index_to_id = meta["index_to_id"]
ntotal = index.ntotal

all_embeddings_matrix = np.array([index.reconstruct(i) for i in range(ntotal)])

faiss_order_df = pd.DataFrame(
    {
        "faiss_idx": list(range(ntotal)),
        "vacancy_id": [index_to_id[i] for i in range(ntotal)],
    }
)

vacancies_df_unique = vacancies_df.drop_duplicates(subset=["vacancy_id"])

df_aligned = faiss_order_df.merge(vacancies_df_unique, on="vacancy_id", how="left")
df_aligned["title"] = df_aligned["title"].fillna("Без названия")

assert len(df_aligned) == len(all_embeddings_matrix), (
    f"Ошибка: строк {len(df_aligned)}, а векторов {len(all_embeddings_matrix)}"
)

In [4]:
kmeans = KMeans(n_clusters=n_anchors, random_state=17)

df_aligned["cluster"] = kmeans.fit_predict(all_embeddings_matrix)

In [5]:
anchors_40 = (
    df_aligned.groupby("cluster").sample(n=1, random_state=17).reset_index(drop=True)
)

remaining_df = df_aligned[~df_aligned["vacancy_id"].isin(anchors_40["vacancy_id"])]
anchors_10_random = remaining_df.sample(
    n=n_random_anchors, random_state=17
).reset_index(drop=True)

final_anchors_df = pd.concat([anchors_40, anchors_10_random], ignore_index=True)

print(f"Собрано {len(final_anchors_df)} вакансий!")

Собрано 50 вакансий!


In [6]:
final_anchors_df.head()

,faiss_idx,vacancy_id,title,author_name,description,city,salary_min,salary_max,requirements,conditions,metro,currency,experience_min,experience_max,tags,remote_type,time_type,author_id,cluster
0,37997,49499010,Эксперт / Ведущий консультант SAP BW / BI,Т1 Консалтинг,T1 Консалтинг внедряет масштабные и удобные ре...,Москва,NaN,NaN,Аналитик,Условия обсуждаются на собеседовании,NaN,RUB,3,6.0,"sap bw, sap bi",OFFICE,FULL,f05c315d-6f65-4400-a342-78d75fc455b1,0
1,22853,49076234,Python-разработчик в SberDevices (Речевая анал...,Сбербанк,SberDevices — молодая IT-компания полного цикл...,Санкт-Петербург,NaN,NaN,"Программист, разработчик",Условия обсуждаются на собеседовании,NaN,RUB,1,3.0,"python, postgresql, linux, sql, ms sql",OFFICE,FULL,61b097c0-26a5-4ef2-9add-a318d26a8390,1
2,896,49596274,Специалист технической поддержки,Мед ИТ-Решения,"Компания МЕД ИТ-Решения (www.medit.ru), один и...",Санкт-Петербург,NaN,NaN,Специалист технической поддержки,Условия обсуждаются на собеседовании,NaN,RUB,0,0.0,"разработка технических заданий, грамотная речь...",OFFICE,FULL,3b4386e8-6802-4bd7-b2bb-e3ffded740e7,2
3,13658,46742113,Системный администратор 1C,Восхождение,Обязанности: Администрирование:1с WMS (склад...,Москва,80000.0,NaN,Системный администратор,Условия обсуждаются на собеседовании,NaN,RUB,1,3.0,"zabbix, linux, bash, ms sql server, ms sql, ст...",OFFICE,FULL,f8ba58fa-6496-4516-8fc3-e43b73beb518,3
4,23236,49072113,Менеджер по работе с клиентами (IT-компания),"Векус, ЦКТ",Наша компания осуществляет: сертифицированное...,Санкт-Петербург,70000.0,150000.0,"Менеджер по продажам, менеджер по работе с кли...",Условия обсуждаются на собеседовании,NaN,RUB,1,3.0,"холодные продажи, активные продажи, ведение пе...",OFFICE,FULL,7295e1a1-67a0-4a44-9ee9-67f4ce732a65,4


In [ ]:
import sys
import os

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.append(project_root)

from src.core.vacancy_rec_sys import VacancyRecSys

d:\Projects\vacancy recommendations\.venv_win\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
rec_sys_e5 = VacancyRecSys()

rec_sys_e5.initialize(initial_df=vacancies_df_unique)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6995.31it/s]


Кэш загружен. Векторов в базе: 47325


In [9]:
tfidf_matrix = sp.load_npz("../data/processed/tfidf_matrix.npz")
df_meta = pd.read_csv("../data/processed/vacancies_meta.csv")

id_to_index = pd.Series(df_meta.index, index=df_meta["vacancy_id"]).to_dict()

In [10]:
def get_recommendations(target_vacancy_id: int, top_k: int = 5) -> pd.DataFrame:

    if target_vacancy_id not in id_to_index:
        return "Ошибка: Вакансия с таким ID не найдена."

    idx = id_to_index[target_vacancy_id]

    target_vector = tfidf_matrix[idx]

    similarities = cosine_similarity(target_vector, tfidf_matrix).flatten()

    # 5. Сортируем индексы, Убираем саму себя из результатов
    related_docs_indices = similarities.argsort()[-2 : -(top_k + 2) : -1]

    result_df = df_meta.iloc[related_docs_indices].copy()

    # Добавляем колонку со score (в процентах)
    result_df["similarity_score"] = similarities[related_docs_indices].round(3)

    return result_df

In [11]:
def format_vacancy_for_llm(row):
    def get_val(col_name, default="Не указано"):
        val = row.get(col_name)
        return str(val).strip() if pd.notna(val) and str(val).strip() != "" else default

    title = get_val("title")
    author = get_val("author_name")
    city = get_val("city")
    remote = get_val("remote_type")
    time_type = get_val("time_type")
    tags = get_val("tags")

    # собираем зарплату
    sal_min = row.get("salary_min")
    sal_max = row.get("salary_max")
    currency = get_val("currency", "")

    if pd.notna(sal_min) and pd.notna(sal_max):
        salary = f"от {int(sal_min)} до {int(sal_max)} {currency}"
    elif pd.notna(sal_min):
        salary = f"от {int(sal_min)} {currency}"
    elif pd.notna(sal_max):
        salary = f"до {int(sal_max)} {currency}"
    else:
        salary = "Не указана"

    # собираем опыт
    exp_min = get_val("experience_min")
    exp_max = get_val("experience_max")
    if exp_min != "Не указано" and exp_max != "Не указано":
        exp = f"от {int(float(exp_min))} до {int(float(exp_max))} лет"
    elif exp_min != "Не указано":
        exp = f"от {int(float(exp_min))} лет"
    else:
        exp = "Без опыта / Не указан"

    # Текстовые блоки
    desc = get_val("description")
    reqs = get_val("requirements")
    # conds = get_val('conditions')

    # сборка промпта-карточки
    return (
        f"Должность: {title}\n"
        f"Компания: {author} ({city})\n"
        f"Формат работы: {time_type}, {remote}\n"
        f"Опыт: {exp}\n"
        f"Зарплата: {salary}\n"
        f"Ключевые навыки: {tags}\n\n"
        f"--- Описание ---\n{desc}\n\n"
        f"--- Требования ---\n{reqs}"
        # f"--- Условия ---\n{conds}"
    )

In [12]:
dataset_rows = []

for _, anchor in final_anchors_df.iterrows():
    anchor_id = anchor["vacancy_id"]
    anchor_text = format_vacancy_for_llm(anchor)

    # Кандидаты от E5
    try:
        e5_result = rec_sys_e5.get_recommendations(
            original_id=anchor_id, top_k=n_e5_candidates
        )
        e5_ids = [item.vacancy_id for item in e5_result.items]
    except ValueError as e:
        print(f"ID {anchor_id} не найден в индексе E5.")
        e5_ids = []

    # Кандидаты от TF-IDF
    tfidf_result = get_recommendations(
        target_vacancy_id=anchor_id, top_k=n_tfidf_candidates
    )

    if isinstance(tfidf_result, pd.DataFrame):
        tfidf_ids = tfidf_result["vacancy_id"].tolist()
    else:
        tfidf_ids = []

    # Случайный шум
    random_candidates = vacancies_df_unique[
        vacancies_df_unique["vacancy_id"] != anchor_id
    ].sample(n_random_candidates, random_state=17)
    random_ids = random_candidates["vacancy_id"].tolist()

    # Дедупликация
    all_candidate_ids = list(set(e5_ids + tfidf_ids + random_ids))

    # Сборка строк для датасета
    for cand_id in all_candidate_ids:
        cand_row = vacancies_df_unique[
            vacancies_df_unique["vacancy_id"] == cand_id
        ].iloc[0]
        cand_text = format_vacancy_for_llm(cand_row)

        found_by = []
        if cand_id in e5_ids:
            found_by.append("E5")
        if cand_id in tfidf_ids:
            found_by.append("TF-IDF")
        if cand_id in random_ids:
            found_by.append("Random")

        dataset_rows.append(
            {
                "anchor_id": anchor_id,
                "anchor_title": anchor["title"],
                "anchor_text": anchor_text,
                "candidate_id": cand_id,
                "candidate_title": cand_row["title"],
                "candidate_text": cand_text,
                "source": " ".join(found_by),
            }
        )

eval_dataset_df = pd.DataFrame(dataset_rows)
print(f"Собрано {len(eval_dataset_df)} уникальных пар для оценки!")

Собрано 2283 уникальных пар для оценки!


In [13]:
# 1. файл со всеми ответами
eval_dataset_df.to_csv(
    "../data/processed/validation/master_eval_dataset.csv",
    index=False,
    encoding="utf-8",
)

# файл для нейронки без колонки 'source'
blind_df = eval_dataset_df[
    [
        "anchor_id",
        "anchor_title",
        "anchor_text",
        "candidate_id",
        "candidate_title",
        "candidate_text",
    ]
].copy()

blind_df["relevance_score"] = ""

blind_df.to_csv(
    "../data/processed/validation/blind_dataset_for_llm.csv",
    index=False,
    encoding="utf-8",
)

print("Файлы сохранены в папку data/processed/validation/")

Файлы сохранены в папку data/processed/validation/


In [14]:
eval_dataset_df['source'].value_counts()

source
E5               784
TF-IDF           783
Random           499
E5 TF-IDF        216
TF-IDF Random      1
Name: count, dtype: int64